# Appendix Table: Annotation Agreement

Computes inter-rater agreement (Cohen's kappa, Fleiss' kappa, Krippendorff's alpha)
between two human annotators and the LLM (Meta-Llama-3.1-405B), per TKGU operation.

Produces the LaTeX table from `s09e_annotation_stats_for_paper.py` in the old pipeline.

**Input:** annotation JSONL file with human + LLM assessments per triple.

In [1]:
import json
import os
import numpy as np
import pandas as pd
from typing import List, Dict
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa
import krippendorff

In [2]:
# --- CONFIGURE THIS ---

# Path to the solved disagreements annotation file
ANNOTATION_PATHS = [
    '/path/to/storage/emerge/output/experiments/s09_annotate_dataset/20250916/solved_disagreements/'
]

HUMAN_ANNOTATORS = [
    {'annotator': 'annotator_1', 'alias': 'H1'},
    {'annotator': 'annotator_2', 'alias': 'H2'},
]

# Which LLMs to use for each prompt type
LLMS_TO_COMPARE = {
    'triple_assertion': {'Meta-Llama-3.1-405B'},
    'triple_deprecation': {'Meta-Llama-3.1-405B'},
}

TKGU_OPERATIONS = ['x-triples', 'e-triples', 'ee-triples', 'ee-kg-triples', 'd-triples']

ACTION_CATEGORY_ASSERT = 'triple_assertion'
ACTION_CATEGORY_DEPRECATE = 'triple_deprecation'

In [3]:
# --- Load and merge annotation data ---

def merge_annotations(instance1, instance2):
    """Merge annotations from two instances of the same hash_id."""
    assert len(instance1['tkgu_triples']) == len(instance2['tkgu_triples'])
    for t1, t2 in zip(instance1['tkgu_triples'], instance2['tkgu_triples']):
        assert t1['triple'] == t2['triple']
        # Merge human assessments
        if 'human_assessment' in t2 and len(t2['human_assessment']) > 0:
            if 'human_assessment' not in t1 or len(t1['human_assessment']) == 0:
                t1['human_assessment'] = t2['human_assessment']
            else:
                existing = {(h['annotator_name'], h['prompt_type']) for h in t1['human_assessment']}
                for h in t2['human_assessment']:
                    if (h['annotator_name'], h['prompt_type']) not in existing:
                        t1['human_assessment'].append(h)
        # Merge LLM assessments
        if 'llm_assessment' in t2 and len(t2['llm_assessment']) > 0:
            if 'llm_assessment' not in t1 or len(t1['llm_assessment']) == 0:
                t1['llm_assessment'] = t2['llm_assessment']
            else:
                existing = {(l['llm_name'], l['llm_prompt_type']) for l in t1['llm_assessment']}
                for l in t2['llm_assessment']:
                    if (l['llm_name'], l['llm_prompt_type']) not in existing:
                        t1['llm_assessment'].append(l)
    return instance1


hash_id_to_instance = {}
for in_dir_path in ANNOTATION_PATHS:
    for fname in os.listdir(in_dir_path):
        fpath = os.path.join(in_dir_path, fname)
        if not fpath.endswith('.jsonl'):
            continue
        for line in open(fpath, 'rt', encoding='utf-8'):
            parsed = json.loads(line)
            hid = parsed['hash_id']
            if hid not in hash_id_to_instance:
                hash_id_to_instance[hid] = parsed
            else:
                hash_id_to_instance[hid] = merge_annotations(
                    hash_id_to_instance[hid], parsed)

print(f'Loaded {len(hash_id_to_instance)} annotated instances')

Loaded 166 annotated instances


In [4]:
# --- Build flat DataFrame of (human, LLM, tkgu) assessment rows ---

def load_annotated_instances(annotated_instances):
    rows = []
    for inst in annotated_instances:
        for triple in inst['tkgu_triples']:
            if 'human_assessment' not in triple or len(triple['human_assessment']) == 0:
                continue
            if 'llm_assessment' not in triple or len(triple['llm_assessment']) == 0:
                continue
            for h_assess in triple['human_assessment']:
                for l_assess in triple['llm_assessment']:
                    if h_assess['prompt_type'] != l_assess['llm_prompt_type']:
                        continue
                    for tkgu_op in triple['tkgu_operations']:
                        # Match prompt type to tkgu operation
                        if (tkgu_op == 'd-triples' and
                                l_assess['llm_prompt_type'] == ACTION_CATEGORY_DEPRECATE):
                            pass  # valid
                        elif (tkgu_op != 'd-triples' and
                                l_assess['llm_prompt_type'] == ACTION_CATEGORY_ASSERT):
                            pass  # valid
                        else:
                            continue
                        rows.append({
                            'hash_id': inst['hash_id'],
                            'human_readable_triple': h_assess['human_readable_triple'],
                            'annotator_name': h_assess['annotator_name'],
                            'llm_name': l_assess['llm_name'],
                            'tkgu_operation': tkgu_op,
                            'llm_assessment': l_assess['llm_assessment'],
                            'human_assessment': h_assess['assessment'],
                        })
    return pd.DataFrame(rows)


df_stats = load_annotated_instances(list(hash_id_to_instance.values()))
print(f'df_stats.shape: {df_stats.shape}')

# Filter to configured LLMs
df_stats = df_stats[
    ((df_stats['tkgu_operation'] == 'd-triples') &
     (df_stats['llm_name'].isin(LLMS_TO_COMPARE['triple_deprecation']))) |
    ((df_stats['tkgu_operation'] != 'd-triples') &
     (df_stats['llm_name'].isin(LLMS_TO_COMPARE['triple_assertion'])))
]

df_stats.rename(columns={'annotator_name': 'human_name'}, inplace=True)
df_stats = df_stats[['human_name', 'human_assessment', 'llm_assessment',
                      'tkgu_operation', 'hash_id', 'human_readable_triple']]
print(f'After LLM filter: {df_stats.shape}')

df_stats.shape: (5644, 7)
After LLM filter: (1411, 6)


In [5]:
# --- Compute agreement statistics ---

human_names = [h['annotator'] for h in HUMAN_ANNOTATORS]
assert len(human_names) == 2

# Keep only triples where both humans annotated and LLM assessment exists
subset = df_stats.groupby(['tkgu_operation', 'hash_id', 'human_readable_triple']).filter(
    lambda x: set(x['human_name']) == set(human_names) and x['llm_assessment'].notna().all()
)

# Pivot humans into columns
human_wide = subset.pivot(
    index=['tkgu_operation', 'hash_id', 'human_readable_triple'],
    columns='human_name',
    values='human_assessment'
)

# Merge LLM assessments
llm_wide = subset.drop_duplicates(
    subset=['tkgu_operation', 'hash_id', 'human_readable_triple']
)[['tkgu_operation', 'hash_id', 'human_readable_triple', 'llm_assessment']].set_index(
    ['tkgu_operation', 'hash_id', 'human_readable_triple']
)

df_wide = human_wide.join(llm_wide).astype(int)

# Per-operation agreement
results = []
for op, group in df_wide.groupby('tkgu_operation'):
    data = group.to_numpy()
    rating_matrix = np.array([np.bincount(row, minlength=2) for row in data])
    results.append({
        'Operation': op,
        'H-H cohen': cohen_kappa_score(group[human_names[0]], group[human_names[1]]),
        'H1-LLM cohen': cohen_kappa_score(group[human_names[0]], group['llm_assessment']),
        'H2-LLM cohen': cohen_kappa_score(group[human_names[1]], group['llm_assessment']),
        'H+LLM fleiss': fleiss_kappa(rating_matrix),
        'H+LLM kripp': krippendorff.alpha(data.T, level_of_measurement='nominal'),
    })

# Overall agreement
data_all = df_wide.to_numpy()
rating_matrix_all = np.array([np.bincount(row, minlength=2) for row in data_all])
results.append({
    'Operation': 'Overall',
    'H-H cohen': cohen_kappa_score(df_wide[human_names[0]], df_wide[human_names[1]]),
    'H1-LLM cohen': cohen_kappa_score(df_wide[human_names[0]], df_wide['llm_assessment']),
    'H2-LLM cohen': cohen_kappa_score(df_wide[human_names[1]], df_wide['llm_assessment']),
    'H+LLM fleiss': fleiss_kappa(rating_matrix_all),
    'H+LLM kripp': krippendorff.alpha(data_all.T, level_of_measurement='nominal'),
})

agreement_table = pd.DataFrame(results)
agreement_table

,Operation,H-H cohen,H1-LLM cohen,H2-LLM cohen,H+LLM fleiss,H+LLM kripp
0,d-triples,0.770531,0.674658,0.609589,0.687134,0.688232
1,e-triples,0.749526,0.697802,0.749526,0.731801,0.732704
2,ee-kg-triples,0.880000,0.840000,0.761146,0.826543,0.827122
3,ee-triples,0.680073,0.810606,0.862888,0.784195,0.784914
4,x-triples,0.717949,0.648649,0.636929,0.668045,0.669151
5,Overall,0.792232,0.764538,0.743608,0.766850,0.767007


## Generate LaTeX table

In [6]:
# Round and reorder
cols = ['H-H cohen', 'H1-LLM cohen', 'H2-LLM cohen', 'H+LLM fleiss', 'H+LLM kripp']
agreement_table[cols] = agreement_table[cols].round(3)

tkgu_type_to_latex = {
    'x-triples': r'\opexists',
    'e-triples': r'\opadd',
    'ee-triples': r'\opmintadd',
    'ee-kg-triples': r'\opinfer',
    'd-triples': r'\opdeprecate',
    'Overall': 'Overall',
}

order = ['x-triples', 'e-triples', 'ee-triples', 'ee-kg-triples', 'd-triples', 'Overall']
agreement_table = (
    agreement_table.set_index('Operation').loc[order].reset_index()
)

# Generate LaTeX rows
latex_rows = []
for _, row in agreement_table.iterrows():
    op_latex = tkgu_type_to_latex.get(row['Operation'], row['Operation'])
    if row['Operation'] == order[-2]:  # last before Overall
        suffix = ' \\\\'
    elif row['Operation'] == 'Overall':
        suffix = ' \\\\'
    else:
        suffix = ' \\\\'
    latex_rows.append(
        f"    {op_latex} & "
        f"{row['H-H cohen']:.3f} & "
        f"{row['H1-LLM cohen']:.3f} & "
        f"{row['H2-LLM cohen']:.3f} & "
        f"{row['H+LLM fleiss']:.3f} & "
        f"{row['H+LLM kripp']:.3f} \\\\"
    )

# Insert midrule before Overall
latex_rows.insert(-1, '    \\midrule')

body = "\n".join(latex_rows)

latex = f"""\\begin{{tabular}}{{lccccc}}
    \\toprule
    \\shortstack{{TKGU \\\\ Operation }} & 
    \\shortstack{{H-H \\\\ Cohen's $\\kappa$}} & 
    \\shortstack{{H1-LLM \\\\ Cohen's $\\kappa$}} & 
    \\shortstack{{H2-LLM \\\\ Cohen's $\\kappa$}} & 
    \\shortstack{{H+LLM \\\\ Fleiss' $\\kappa$}} & 
    \\shortstack{{H+LLM \\\\ Kripp. $\\alpha$}} \\\\
    \\midrule
{body}
    \\bottomrule
    \\end{{tabular}}"""

print(latex)

\begin{tabular}{lccccc}
    \toprule
    \shortstack{TKGU \\ Operation } & 
    \shortstack{H-H \\ Cohen's $\kappa$} & 
    \shortstack{H1-LLM \\ Cohen's $\kappa$} & 
    \shortstack{H2-LLM \\ Cohen's $\kappa$} & 
    \shortstack{H+LLM \\ Fleiss' $\kappa$} & 
    \shortstack{H+LLM \\ Kripp. $\alpha$} \\
    \midrule
    \opexists & 0.718 & 0.649 & 0.637 & 0.668 & 0.669 \\
    \opadd & 0.750 & 0.698 & 0.750 & 0.732 & 0.733 \\
    \opmintadd & 0.680 & 0.811 & 0.863 & 0.784 & 0.785 \\
    \opinfer & 0.880 & 0.840 & 0.761 & 0.827 & 0.827 \\
    \opdeprecate & 0.771 & 0.675 & 0.610 & 0.687 & 0.688 \\
    \midrule
    Overall & 0.792 & 0.765 & 0.744 & 0.767 & 0.767 \\
    \bottomrule
    \end{tabular}
